# The Heritage Short — Music Score Generator (ACE-Step 1.5)

**Kaggle settings:** Accelerator = **GPU T4 x2** (or T4), **Internet = On**.

Generates **3 instrumental score candidates** (`60 s`, `72 BPM`, `D major`, seeds `777, 778, 779`) matching `storyboard.json` -> `music`, plus `music_run_metadata.json` for `py run.py music`.

In [ ]:
# 1. Install ACE-Step 1.5 (pinned to main repo; T4-safe PyTorch backend)
!git clone --depth 1 https://github.com/ACE-Step/ACE-Step-1.5.git /kaggle/working/ACE-Step-1.5
%cd /kaggle/working/ACE-Step-1.5
!pip install -q -e . --no-build-isolation
!pip install -q soundfile

In [ ]:
# 2. Music spec (copied directly from storyboard.json -> music)
import json, time, shutil
from pathlib import Path

MUSIC_SPEC = {
    "model": "ACE-Step/Ace-Step1.5",
    "dit_config": "acestep-v15-turbo",
    "lm_model": "acestep-5Hz-lm-0.6B",
    "caption": "Warm acoustic Ethiopian-inspired folk instrumental with gentle krar strings, soft bamboo flute, and subtle felt piano, unhurried morning courtyard mood, organic warm texture, no vocals",
    "style": "acoustic folk instrumental",
    "mood": "warm, grateful, unhurried, nostalgic",
    "instruments": "krar plucked strings, gentle bamboo flute, soft felt piano",
    "texture": "warm organic acoustic",
    "bpm": 72,
    "keyscale": "D major",
    "duration": 60.0,
    "base_seed": 777,
    "num_candidates": 3,
}

OUT_DIR = Path("/kaggle/working/music_candidates")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Ready. Output directory:", OUT_DIR)

In [ ]:
# 3. Initialize ACE-Step 1.5 (PyTorch backend for reliable T4 compatibility)
from acestep.handler import AceStepHandler
from acestep.llm_inference import LLMHandler
from acestep.inference import GenerationParams, GenerationConfig, generate_music

PROJECT_ROOT = "/kaggle/working/ACE-Step-1.5"
CHECKPOINT_DIR = f"{PROJECT_ROOT}/checkpoints"

dit_handler = AceStepHandler()
dit_handler.initialize_service(
    project_root=PROJECT_ROOT,
    config_path=MUSIC_SPEC["dit_config"],
    device="cuda",
)

llm_handler = LLMHandler()
llm_handler.initialize(
    checkpoint_dir=CHECKPOINT_DIR,
    lm_model_path=MUSIC_SPEC["lm_model"],
    backend="pt",
    device="cuda",
)
print("Models initialized.")

In [ ]:
# 4. Generate 3 instrumental candidates sequentially (fixed seeds 777, 778, 779)
records = []
for idx in range(MUSIC_SPEC["num_candidates"]):
    seed = MUSIC_SPEC["base_seed"] + idx
    t0 = time.perf_counter()
    params = GenerationParams(
        task_type="text2music",
        caption=MUSIC_SPEC["caption"],
        lyrics="",
        instrumental=True,
        bpm=MUSIC_SPEC["bpm"],
        keyscale=MUSIC_SPEC["keyscale"],
        duration=MUSIC_SPEC["duration"],
        inference_steps=8,
        seed=seed,
    )
    config = GenerationConfig(
        batch_size=1,
        use_random_seed=False,
        seeds=[seed],
        audio_format="wav",
    )
    result = generate_music(
        dit_handler,
        llm_handler,
        params,
        config,
        save_dir=str(OUT_DIR / f"tmp_{idx+1}"),
    )
    latency_ms = int((time.perf_counter() - t0) * 1000)
    if not result.success or not result.audios:
        raise RuntimeError(f"Candidate {idx+1} (seed={seed}) failed: {result.error}")

    src_path = Path(result.audios[0]["path"])
    dst_name = f"candidate_{idx+1:02d}_seed{seed}.wav"
    dst_path = OUT_DIR / dst_name
    shutil.copy2(src_path, dst_path)
    records.append({
        "candidate_id": f"candidate_{idx+1:02d}",
        "file_name": dst_name,
        "model": MUSIC_SPEC["model"],
        "dit_config": MUSIC_SPEC["dit_config"],
        "seed": seed,
        "bpm": MUSIC_SPEC["bpm"],
        "keyscale": MUSIC_SPEC["keyscale"],
        "duration_sec": MUSIC_SPEC["duration"],
        "latency_ms": latency_ms,
        "status": "ok",
    })
    print(f"Saved {dst_name} in {latency_ms} ms")

meta_path = OUT_DIR / "music_run_metadata.json"
meta_path.write_text(json.dumps({"spec": MUSIC_SPEC, "candidates": records}, indent=2), encoding="utf-8")
print("Wrote", meta_path)

In [ ]:
# 5. Package all 3 WAVs + metadata into one downloadable zip
import IPython.display as ipd

zip_path = shutil.make_archive("/kaggle/working/heritage_music_candidates", "zip", OUT_DIR)
print("Download this zip and extract into outputs/music/:", zip_path)
for rec in records:
    print(rec["file_name"], f"({rec['latency_ms']} ms)")
    ipd.display(ipd.Audio(str(OUT_DIR / rec["file_name"])))